# Global → local spatial autocorrelation

Global Moran's I ranking (validated against the from-scratch `moran_scratch`), then LISA + Gi\* on the top-ranked genes, FDR within a gene, isolate masking, and hotspot overlays on the aligned H&E with the compartment-recovery check.

## Smoke test — ranking + scratch parity

The from-scratch Moran's I must match squidpy on the same row-standardized weights before the ranking is trusted.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pathlib, sys
import numpy as np

# Offline smoke data: the project's committed synthetic fixture (no download).
_root = pathlib.Path.cwd()
_root = _root if (_root / "pyproject.toml").exists() else _root.parent
sys.path.insert(0, str(_root / "tests" / "fixtures"))
from synthetic import make_synthetic_visium

from visium_spatial.build_graph import build_spatial_graph, to_libpysal_weights
from visium_spatial.global_autocorr import rank_svgs, top_genes
from visium_spatial.moran_scratch import morans_i

adata = make_synthetic_visium(seed=0)
build_spatial_graph(adata)
ranked = rank_svgs(adata, seed=0)

W = to_libpysal_weights(adata, transform="r")
W_dense = W.full()[0]
for g in adata.var_names:
    x = np.asarray(adata[:, g].X).ravel().astype(float)
    assert np.isclose(ranked.loc[g, "I"], morans_i(x, W_dense), atol=1e-6)
print("scratch <-> squidpy parity: OK")
ranked

## Local indicators + hotspot overlays

LISA (HH/LH/LL/HL) and Gi\* on the top-ranked genes, with per-spot FDR within a gene and isolate masking, drawn over the tissue via the scalefactor. On a real section pass the hires H&E array as `image=` and get `SCALEF` from `extract_scalefactors`.

In [ ]:
import matplotlib.pyplot as plt
from visium_spatial.local_autocorr import local_moran, getis_ord_gi, lisa_quadrants
from visium_spatial.multitest import fdr_within_gene, mask_isolates
from visium_spatial.overlay import plot_hotspots

top = top_genes(ranked, n=2, max_pval=0.05)
SCALEF = 0.15  # tissue_hires_scalef; from extract_scalefactors on a real section

for g in top.index:
    x = np.asarray(adata[:, g].X).ravel().astype(float)
    lm = local_moran(x, W, seed=0)
    _, q = fdr_within_gene(mask_isolates(lm.p_sim, W))   # isolates -> NaN -> excluded
    labels = lisa_quadrants(lm, p_thresh=0.05, pvals=q)
    gi = getis_ord_gi(x, W, seed=0)

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(10, 4))
    plot_hotspots(adata, labels, scalefactor=SCALEF, ax=axL, title=f"{g} — LISA")
    plot_hotspots(adata, gi.Zs, scalefactor=SCALEF, ax=axG, title=f"{g} — Gi* z")
    plt.show()

# compartment-recovery check: the planted patch should come back High-High
lm_hh = local_moran(np.asarray(adata[:, "GENE_HH"].X).ravel().astype(float), W, seed=0)
hh_labels = lisa_quadrants(lm_hh, p_thresh=0.05)
print("GENE_HH High-High spots:", int((hh_labels == "HH").sum()))

## Real run

Load the section as in the EDA notebook, then repeat the two cells above with the real `SCALEF` and the hires H&E passed as `image=` to `plot_hotspots`. Check whether High-High clusters for known compartment markers recover known architecture (follicle markers -> follicles, T-zone markers -> paracortex); cite marker sources.